In [2]:
import time
waktu = {}

def mulai():
    global _t
    _t = time.perf_counter()

def selesai(nama):
    waktu[nama] = time.perf_counter() - _t
    print(f"⏱️ {nama}: {waktu[nama]:.2f} detik")

def ringkasan(judul, tahap_analisis=None):
    print(f"\n=== RINGKASAN WAKTU — {judul} ===")
    for n, d in waktu.items():
        print(f"{n:30s}: {d:8.2f} detik")
    if tahap_analisis:
        total = sum(waktu.get(t, 0) for t in tahap_analisis)
        print(f"\n>> Total waktu analisis: {total:.2f} detik")

In [3]:
# TAHAP 1: Kloning ke Disk Lokal & Mount Drive
mulai()
import os
from google.colab import drive

# 1. Mount Google Drive untuk penyimpanan hasil akhir
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/Riset_Moodle'
os.makedirs(DRIVE_PATH, exist_ok=True)

# 2. Kloning Moodle langsung ke SSD lokal Colab
REPO_PATH = '/content/moodle'
if not os.path.exists(REPO_PATH):
    print("Mengunduh source code Moodle ke disk lokal Colab...")
    !git clone https://github.com/moodle/moodle.git {REPO_PATH} --depth 1
    print("Kloning selesai! 100% file fisik siap dipindai.")
else:
    print("Source code lokal sudah tersedia. Aman!")
selesai("Fase1 - Kloning")

Mounted at /content/drive
Mengunduh source code Moodle ke disk lokal Colab...
Cloning into '/content/moodle'...
remote: Enumerating objects: 71642, done.
remote: Counting objects: 100% (71642/71642), done.
remote: Compressing objects: 100% (34937/34937), done.
remote: Total 71642 (delta 33591), reused 68317 (delta 33035), pack-reused 0 (from 0)
Receiving objects: 100% (71642/71642), 95.05 MiB | 15.78 MiB/s, done.
Resolving deltas: 100% (33591/33591), done.
Updating files: 100% (61794/61794), done.
Kloning selesai! 100% file fisik siap dipindai.
⏱️ Fase1 - Kloning: 41.69 detik


In [11]:
import glob, re

# Cari file version.php-nya dulu
paths = glob.glob('/content/moodle/version.php')
if not paths:
    paths = glob.glob('/content/moodle/**/version.php', recursive=True)
print("Lokasi file:", paths[:3])

if paths:
    isi = open(paths[0]).read()
    for var in ['release', 'branch', 'version', 'maturity']:
        m = re.search(rf"\${var}\s*=\s*'?([^;'\n]+)", isi)
        print(f"${var:8s}= {m.group(1).strip() if m else '(tidak ketemu)'}")
else:
    print("version.php tidak ditemukan — cek isi folder:")
    !ls /content/moodle | head -20

Lokasi file: ['/content/moodle/public/version.php', '/content/moodle/public/filter/multilang/version.php', '/content/moodle/public/filter/tex/version.php']
$release = 5.3dev (Build: 20260616)
$branch  = 503
$version = 2026061600.00
$maturity= MATURITY_ALPHA


In [4]:
# TAHAP 2: Pemetaan Struktur Direktori (Node List)
mulai()
import pandas as pd

data_artefak = []
print("Memetakan struktur direktori lokal...")

for root, dirs, files in os.walk(REPO_PATH):
    # Bypass folder dependensi eksternal agar murni menganalisis kode Moodle
    if '.git' in root or 'vendor' in root or 'node_modules' in root:
        continue

    for file in files:
        if file.endswith('.php'): # Kita hanya peduli pada file PHP
            file_path = os.path.join(root, file)
            rel_path = os.path.relpath(file_path, start=REPO_PATH).replace('\\', '/')
            path_parts = rel_path.split('/')

            # Logika Identifikasi Modul Dasar
            nama_modul = 'N/A'
            if path_parts[0] == 'mod' and len(path_parts) > 1:
                nama_modul = path_parts[1]

            data_artefak.append({
                'Path_Lengkap': rel_path,
                'Nama_Modul': nama_modul
            })

df_php = pd.DataFrame(data_artefak)
print(f"Pemetaan selesai! Ditemukan {len(df_php)} file PHP yang siap ditambang.")
selesai("Fase1 - Pemetaan")

Memetakan struktur direktori lokal...
Pemetaan selesai! Ditemukan 50028 file PHP yang siap ditambang.
⏱️ Fase1 - Pemetaan: 1.37 detik


In [5]:
# TAHAP 3: Fungsi Mesin Ekstraksi (Regex untuk Include & Function Call)
import re

# Blacklist kata kunci bawaan PHP agar tidak dianggap sebagai fungsi Moodle
PHP_KEYWORDS = {'if', 'elseif', 'else', 'while', 'for', 'foreach', 'switch', 'array', 'isset', 'empty', 'eval', 'exit', 'die', 'list', 'echo', 'print', 'catch', 'function', 'return', 'include', 'require'}

def baca_file_php(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            return f.read()
    except Exception: return None

def ekstrak_dependensi(konten_teks):
    list_dependensi = []

    # --- 1. Regex untuk Pemanggilan File (Include/Require) ---
    pola_file = r'\b(require_once|require|include_once|include)\s*\(?\s*(.*?)\s*\)?\s*;'
    for tipe, path in re.findall(pola_file, konten_teks, re.IGNORECASE):
        path_bersih = path.replace("'", "").replace('"', "").replace(" . ", "").strip()
        path_bersih = path_bersih.replace("$CFG->dirroot.", "").replace("$CFG->dirroot", "")
        path_bersih = path_bersih.replace("$CFG->libdir", "/lib")
        path_bersih = re.sub(r'/+', '/', path_bersih).lstrip('/')
        if '$' in path_bersih: path_bersih = "[DYNAMIC] " + path_bersih

        list_dependensi.append({'Tipe_Dependensi': tipe, 'File_Tujuan': path_bersih})

    # --- 2. Regex untuk Pemanggilan Fungsi ---
    pola_fungsi = r'\b([a-zA-Z_\x7f-\xff][a-zA-Z0-9_\x7f-\xff]*)\s*\('
    for nama_fungsi in re.findall(pola_fungsi, konten_teks):
        if nama_fungsi.lower() not in PHP_KEYWORDS:
            list_dependensi.append({
                'Tipe_Dependensi': 'function_call',
                'File_Tujuan': f"Fungsi: {nama_fungsi}()"
            })

    return list_dependensi

In [6]:
# TAHAP 4: Eksekusi Penambangan Data (Data Mining)
mulai()
data_hasil_scan = []

print("Mulai menambang data dari puluhan ribu file. Silakan tunggu, ini butuh waktu...")

for index, row in df_php.iterrows():
    file_sumber = row['Path_Lengkap']

    path_fisik = os.path.join(REPO_PATH, file_sumber)
    konten = baca_file_php(path_fisik)

    if konten:
        dependensi = ekstrak_dependensi(konten)
        for dep in dependensi:
            data_hasil_scan.append({
                'File_Sumber': file_sumber,
                'File_Tujuan': dep['File_Tujuan'],
                'Tipe_Dependensi': dep['Tipe_Dependensi']
            })

df_dependensi = pd.DataFrame(data_hasil_scan)
print(f"\nPenambangan Selesai! Berhasil mengekstrak {len(df_dependensi)} baris relasi.")
selesai("Fase1 - Penambangan")

Mulai menambang data dari puluhan ribu file. Silakan tunggu, ini butuh waktu...

Penambangan Selesai! Berhasil mengekstrak 1247184 baris relasi.
⏱️ Fase1 - Penambangan: 42.83 detik


In [8]:
# TAHAP 5: Ekspor Hasil ke Google Drive
mulai()
OUTPUT_FILE = f'{DRIVE_PATH}/moodle_dependencies_full.csv'
df_dependensi.to_csv(OUTPUT_FILE, index=False)

print("=== RINGKASAN DATA MINING ===")
print(df_dependensi['Tipe_Dependensi'].value_counts())
print(f"\nDataset utama berhasil diamankan secara permanen di: {OUTPUT_FILE}")
selesai("Fase1 - Eksport")
ringkasan("FASE 1", tahap_analisis=["Fase1 - Pemetaan", "Fase1 - Penambangan"])

=== RINGKASAN DATA MINING ===
Tipe_Dependensi
function_call    1227784
require             8702
require_once        8340
include             1674
REQUIRE              305
include_once         169
Require              104
Include               85
INCLUDE               21
Name: count, dtype: int64

Dataset utama berhasil diamankan secara permanen di: /content/drive/MyDrive/Riset_Moodle/moodle_dependencies_full.csv
⏱️ Fase1 - Eksport: 7.05 detik

=== RINGKASAN WAKTU — FASE 1 ===
Fase1 - Kloning               :    41.69 detik
Fase1 - Pemetaan              :     1.37 detik
Fase1 - Penambangan           :    42.83 detik
Fase1 - Eksport               :     7.05 detik

>> Total waktu analisis: 44.19 detik
